<a href="https://colab.research.google.com/github/ChanZH0525/Salary-Prediction-Prototype-Model/blob/main/new_xgboost_salary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q xgboost scikit-learn pandas numpy datasets category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 1.5 MB/s eta 0:00:00


In [48]:
import os
import random
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.sparse import hstack, csr_matrix

from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datasets import load_dataset

SEED = 42

def set_all_seeds(seed: int = SEED):
    """Comprehensive seeding, mirroring the discipline used for the JobBERT runs
    (Python, NumPy, and here XGBoost's own seed params) for replicability."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)

In [49]:
df = pd.read_csv('/content/df_labeled_cleaned_with_exp.csv')
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28960 entries, 0 to 28959
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   job_id              28960 non-null  int64  
 1   company             28960 non-null  object 
 2   type                28960 non-null  object 
 3   salary              28960 non-null  object 
 4   listingDate         28960 non-null  object 
 5   descriptions_clean  28960 non-null  object 
 6   desc_word_len       28960 non-null  int64  
 7   job_title_clean     28960 non-null  object 
 8   role_clean          28960 non-null  object 
 9   state               28960 non-null  object 
 10  salary_range        28960 non-null  object 
 11  salary_mid          28960 non-null  float64
 12  salary_width        28960 non-null  float64
 13  salary_min          28960 non-null  float64
 14  salary_max          28960 non-null  float64
 15  category_celan      28960 non-null  object 
 16  subc

,job_id,company,type,salary,listingDate,descriptions_clean,desc_word_len,job_title_clean,role_clean,state,salary_range,salary_mid,salary_width,salary_min,salary_max,category_celan,subcategory_celan,exp_min,exp_max,experience_raw
0,84094132,VERONA BAKE (M) SDN BHD,Full time,"RM 3,500 – RM 4,500 per month",8/5/2025 10:17,We are a beloved bakery brand known for our fr...,296,Marketing Executive,marketing executive,Selangor,"3,500 - 4,500",4000.0,1000.0,3500.0,4500.0,marketing & communications,marketing communications,1.0,2.0,1 to 2
1,84090140,SRG Asia Pacific Sdn. Bhd.,Full time,"RM 4,000 – RM 4,500 per month",8/5/2025 10:10,POSITION SUMMARY We are seeking a detail-orien...,715,Customer Service Administrator | Import Export,customer service administrator,Selangor,"4,000 - 4,500",4250.0,500.0,4000.0,4500.0,"manufacturing, transport & logistics",import/export & customs,NaN,NaN,Unspecified
2,84092513,Kanry Search,Full time,"RM 4,000 – RM 6,000 per month",8/5/2025 9:25,JOB RESPONSIBILITIES Collaborate with the Plan...,291,Buyer,buyer,Johor,"4,000 - 6,000",5000.0,2000.0,4000.0,6000.0,"manufacturing, transport & logistics","purchasing, procurement & inventory",NaN,NaN,Unspecified
3,84091988,TT Freight Group Sdn Bhd,Full time,"RM 3,000 – RM 4,500 per month",8/5/2025 9:06,Responsibility Building and maintaining relati...,219,Sales Executive,sales executive,Selangor,"3,000 - 4,500",3750.0,1500.0,3000.0,4500.0,sales,sales representatives/consultants,0.0,0.0,0
4,84091161,VILLAGE GROCER HOLDINGS SDN BHD,Full time,"RM 1,700 – RM 2,400 per month",8/5/2025 8:45,Job Description Assistant Baker Strictly follo...,196,Assistant Baker (Central Bakery Kitchen),bakery assistant,Selangor,"1,700 - 2,400",2050.0,700.0,1700.0,2400.0,trades & services,bakers & pastry chefs,1.0,3.0,1 to 3


In [50]:
# Manual correction: raw "RM 7000 - RM 11 000" was misparsed due to a space
# used as a thousands separator in "11 000" (instead of comma/no separator),
# causing the parser to truncate it to 11 and misassign min/max as 0/7000.
# Confirmed via df['salary'].str.contains(r'\d \d{3}\b') that this is the
# only row in the dataset affected by this specific space-separator pattern.

mask = df["job_id"] ==  74018456 # fill in the actual job_id for this row

df.loc[mask, "salary_min"] = 7000.0
df.loc[mask, "salary_max"] = 11000.0
df.loc[mask, "salary_mid"] = (7000.0 + 11000.0) / 2
df.loc[mask, "salary_width"] = 11000.0 - 7000.0
df.loc[mask, "salary_range"] = "7,000 - 11,000"

print(df.loc[mask, ["salary", "salary_min", "salary_max", "salary_mid", "salary_width", "salary_range"]])

                    salary  salary_min  salary_max  salary_mid  salary_width  \
28137  RM 7000 - RM 11 000      7000.0     11000.0      9000.0        4000.0   

         salary_range  
28137  7,000 - 11,000  


In [51]:
# Stage 1, step 9 (revised): upper salary_mid bound lowered from RM 150,000
# to RM 20,000. Bracket-level row counts (fit on train, RM 2,000 bins)
# showed brackets above RM 20,000 routinely contain 0-9 rows per split,
# too sparse to support reliable model training or evaluation. This trims
# ~147 rows (~0.5% of the dataset) previously retained under the original
# [RM 500, RM 150,000] bound.
SALARY_MID_UPPER_BOUND = 20000
MID_COL = "salary_mid"

before_n = len(df)
df = df[df[MID_COL] <= SALARY_MID_UPPER_BOUND].reset_index(drop=True)
print(f"Dropped {before_n - len(df)} rows with salary_mid > RM {SALARY_MID_UPPER_BOUND:,} "
      f"({before_n} -> {len(df)})")

Dropped 138 rows with salary_mid > RM 20,000 (28960 -> 28822)


In [52]:
DATE_COLUMN = "listingDate"

# Format is D/M/YYYY H:MM (e.g. "8/5/2025 10:17"), day-first, must be explicit,
# since day <= 12 rows are ambiguous and pandas silently mis-infers month-first.
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], format="%d/%m/%Y %H:%M", errors="raise")

df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

# Calendar-based split: each month belongs to exactly one set.
TRAIN_START = "2024-01-01"
VAL_START   = "2024-06-01"
TEST_START  = "2025-01-01"

train_df = df[(df[DATE_COLUMN] >= TRAIN_START) & (df[DATE_COLUMN] < VAL_START)].copy()
val_df   = df[(df[DATE_COLUMN] >= VAL_START)   & (df[DATE_COLUMN] < "2024-07-01")].copy()
test_df  = df[df[DATE_COLUMN] >= TEST_START].copy()

n_used = len(train_df) + len(val_df) + len(test_df)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Split shares: {len(train_df)/n_used:.1%} / {len(val_df)/n_used:.1%} / {len(test_df)/n_used:.1%}")
print(f"Train dates: {train_df[DATE_COLUMN].min().date()} to {train_df[DATE_COLUMN].max().date()}")
print(f"Val dates:   {val_df[DATE_COLUMN].min().date()} to {val_df[DATE_COLUMN].max().date()}")
print(f"Test dates:  {test_df[DATE_COLUMN].min().date()} to {test_df[DATE_COLUMN].max().date()}")

# No rows should fall outside the three windows, and no window should overlap.
assert n_used == len(df), f"{len(df) - n_used} rows fall outside the split windows"
assert train_df[DATE_COLUMN].max() < val_df[DATE_COLUMN].min()
assert val_df[DATE_COLUMN].max() < test_df[DATE_COLUMN].min()

Train: 20462 | Val: 3660 | Test: 4700
Split shares: 71.0% / 12.7% / 16.3%
Train dates: 2024-01-27 to 2024-05-31
Val dates:   2024-06-01 to 2024-06-19
Test dates:  2025-03-03 to 2025-05-08


In [53]:
print("Train:", (train_df["salary_max"] < train_df["salary_min"]).sum())
print(train_df[["salary_min","salary_max","salary_mid"]].describe())

Train: 0
         salary_min    salary_max    salary_mid
count  20462.000000  20462.000000  20462.000000
mean    3870.164110   5246.287523   4558.225817
std     2210.103704   3018.269599   2599.788089
min      400.000000    500.000000    500.000000
25%     2500.000000   3500.000000   3000.000000
50%     3300.000000   4500.000000   3750.000000
75%     4500.000000   6000.000000   5150.000000
max    20000.000000  25000.000000  20000.000000


In [54]:
import ast

def clean_type(val):
    try:
        parsed = ast.literal_eval(val) if isinstance(val, str) and val.startswith("[") else val
        if isinstance(parsed, list):
            return parsed[0] if parsed else "missing"
        return str(parsed)
    except (ValueError, SyntaxError):
        return str(val)

# Diagnostic: how many rows per split were actually list-strings before cleaning?
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n_list_strings = d["type"].astype(str).str.startswith("[").sum()
    n_multi_value = d["type"].astype(str).apply(
        lambda v: v.startswith("[") and len(ast.literal_eval(v)) > 1 if v.startswith("[") else False
    ).sum()
    print(f"{name}: {n_list_strings} list-string rows, {n_multi_value} with >1 value (first-value-only kept)")

train_df["type"] = train_df["type"].apply(clean_type)
val_df["type"] = val_df["type"].apply(clean_type)
test_df["type"] = test_df["type"].apply(clean_type)

print("\nPost-clean type value counts (train):")
print(train_df["type"].value_counts())

train: 0 list-string rows, 0 with >1 value (first-value-only kept)
val: 0 list-string rows, 0 with >1 value (first-value-only kept)
test: 0 list-string rows, 0 with >1 value (first-value-only kept)

Post-clean type value counts (train):
type
Full time        19203
Contract/Temp     1259
Name: count, dtype: int64


In [55]:
import re

CATEGORICAL_COLS = ["category_celan", "subcategory_celan", "role_clean", "state", "type"]
EXPERIENCE_COLS = ["exp_min", "exp_max"]  # raw NaN preserved — XGBoost splits natively on missingness

# ----------------------------------------------------------------------
# Keyword matching: WORD-BOUNDARY regex, not substring.
# Substring matching previously made "coo" hit coordinator/cook and "cto"
# hit director/factory/doctor/inspector (all scored as C-suite), and
# "intern" hit international/internal.
# ----------------------------------------------------------------------
# Special-case patterns; everything else falls back to \b<kw>s?\b
_SPECIAL_PATTERNS = {
    "intern": r"\b(?:intern|interns|internship|internships)\b",
    "lead": r"\b(?:lead|leader|leaders)\b",
    "entry": r"(?<!data )\bentry(?:[- ]level)?\b",   # excludes "data entry"
    "head": r"\bhead\b",
}

def kw_pattern(kw: str) -> str:
    if kw in _SPECIAL_PATTERNS:
        return _SPECIAL_PATTERNS[kw]
    return rf"\b{re.escape(kw)}s?\b"

SENIORITY_LEVELS = [
    (0, ["intern", "trainee"]),
    (1, ["junior", "entry", "assistant", "clerk", "fresh"]),
    (2, ["executive", "officer", "associate", "coordinator", "specialist"]),
    (3, ["senior", "sr", "lead", "principal", "supervisor", "analyst"]),
    (4, ["manager", "head", "controller"]),
    (5, ["director", "gm", "general manager", "vp", "vice president", "avp"]),
    (6, ["chief", "ceo", "coo", "cfo", "cto", "president", "founder", "co-founder"]),
]
DEFAULT_SENIORITY_SCORE = 2  # mid-level default when no keyword matches

# Precompile one regex per level
_LEVEL_REGEX = [
    (level, re.compile("|".join(kw_pattern(kw) for kw in kws)))
    for level, kws in SENIORITY_LEVELS
]

def seniority_score(text: str) -> int:
    text = text.lower()
    matched = [level for level, rx in _LEVEL_REGEX if rx.search(text)]
    return max(matched) if matched else DEFAULT_SENIORITY_SCORE

SENIORITY_KEYWORDS = [
    "senior", "lead", "head", "manager", "director", "executive",
    "chief", "vp", "vice president", "principal", "junior", "intern",
    "entry", "gm", "controller", "ceo", "founder",
    "assistant", "clerk", "officer", "associate", "coordinator",
    "specialist", "supervisor", "analyst", "fresh",
]
_KW_REGEX = {kw: kw_pattern(kw) for kw in SENIORITY_KEYWORDS}

def engineer_text_stats(df: pd.DataFrame, col: str, prefix: str) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    text = df[col].fillna("").astype(str)

    out[f"{prefix}_len"] = text.str.len()
    out[f"{prefix}_word_count"] = text.str.split().str.len()

    lower = text.str.lower()
    for kw in SENIORITY_KEYWORDS:
        col_name = f"{prefix}_kw_" + kw.replace(" ", "_")
        out[col_name] = lower.str.contains(_KW_REGEX[kw], regex=True).astype(int)
    out[f"{prefix}_seniority_score"] = lower.apply(seniority_score)
    return out

# ----------------------------------------------------------------------
# Sanity check for the keyword bug — all of these must pass.
# ----------------------------------------------------------------------
_expected = {
    "Marketing Coordinator": 2,        # was 6 (coo in coordinator)
    "Cook": 2,                         # was 6
    "Factory Operator": 2,             # was 6 (cto in factory)
    "Quality Inspector": 2,            # was 6
    "Sales Director": 5,               # was 6
    "International Sales Executive": 2,
    "Data Entry Clerk": 1,             # clerk only; "data entry" not entry-level
    "Senior Software Engineer": 3,
    "Assistant Manager": 4,
    "Chief Executive Officer": 6,
    "Intern": 0,
}
for title, exp_score in _expected.items():
    got = seniority_score(title)
    assert got == exp_score, f"{title!r}: expected {exp_score}, got {got}"
_t = engineer_text_stats(pd.DataFrame({"t": ["International Sales Executive", "Software Intern"]}), "t", "title")
assert _t["title_kw_intern"].tolist() == [0, 1], "intern flag still matching 'international'"

# ----------------------------------------------------------------------
# Rare / unseen category handling, TRAIN frequency only.
# role_clean: collapse values seen < ROLE_MIN_FREQ times in train into "other".
# subcategory: collapse values never seen in train into "other"
# (val/test previously produced NaN for these under set_categories).
# ----------------------------------------------------------------------
ROLE_MIN_FREQ = 20

train_role_counts = train_df["role_clean"].fillna("missing").astype(str).value_counts()
valid_roles = set(train_role_counts[train_role_counts >= ROLE_MIN_FREQ].index)

valid_subcats = set(train_df["subcategory_celan"].fillna("missing").astype(str))

def bucket_rare_roles(df: pd.DataFrame) -> pd.Series:
    role_str = df["role_clean"].fillna("missing").astype(str)
    return role_str.where(role_str.isin(valid_roles), "other")

def bucket_unseen_subcats(df: pd.DataFrame) -> pd.Series:
    sub_str = df["subcategory_celan"].fillna("missing").astype(str)
    return sub_str.where(sub_str.isin(valid_subcats), "other")

def build_tabular_features(df: pd.DataFrame) -> pd.DataFrame:
    cat_df = df[CATEGORICAL_COLS].fillna("missing").astype(str)
    cat_df["role_clean"] = bucket_rare_roles(df)
    cat_df["subcategory_celan"] = bucket_unseen_subcats(df)
    cat_df = cat_df.astype("category")
    cat_df.columns = [f"{c}_enc" for c in CATEGORICAL_COLS]

    title_feats = engineer_text_stats(df, "job_title_clean", "title")

    # exp_min / exp_max: raw numeric, NaN preserved (no imputation).
    exp_feats = df[EXPERIENCE_COLS].copy()

    return pd.concat([cat_df, title_feats, exp_feats], axis=1)

X_train_tab = build_tabular_features(train_df)
X_val_tab = build_tabular_features(val_df)
X_test_tab = build_tabular_features(test_df)

# Align category levels to train so val/test never see unknown levels.
# "other" must exist in train levels for unseen values to map to it.
for c in [f"{col}_enc" for col in CATEGORICAL_COLS]:
    train_cats = X_train_tab[c].cat.categories
    if "other" not in train_cats:
        train_cats = train_cats.append(pd.Index(["other"]))
        X_train_tab[c] = X_train_tab[c].cat.set_categories(train_cats)
    X_val_tab[c] = X_val_tab[c].cat.set_categories(train_cats)
    X_test_tab[c] = X_test_tab[c].cat.set_categories(train_cats)

tfidf = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),
    min_df=50,
    lowercase=True,
)

title_train_tfidf = tfidf.fit_transform(train_df["job_title_clean"].fillna("").astype(str))
title_val_tfidf = tfidf.transform(val_df["job_title_clean"].fillna("").astype(str))
title_test_tfidf = tfidf.transform(test_df["job_title_clean"].fillna("").astype(str))

tfidf_feature_names = [f"title_tfidf_{t}" for t in tfidf.get_feature_names_out()]

title_train_tfidf_df = pd.DataFrame(title_train_tfidf.toarray(), columns=tfidf_feature_names, index=train_df.index)
title_val_tfidf_df = pd.DataFrame(title_val_tfidf.toarray(), columns=tfidf_feature_names, index=val_df.index)
title_test_tfidf_df = pd.DataFrame(title_test_tfidf.toarray(), columns=tfidf_feature_names, index=test_df.index)

X_train = pd.concat([X_train_tab, title_train_tfidf_df], axis=1)
X_val = pd.concat([X_val_tab, title_val_tfidf_df], axis=1)
X_test = pd.concat([X_test_tab, title_test_tfidf_df], axis=1)

all_feature_names = list(X_train.columns)

# Expected: 5 categorical + 29 title (len, word_count, 26 keyword flags, seniority)
#           + 2 experience = 36 tabular; + 200 TF-IDF = 236 total.
n_title = 2 + len(SENIORITY_KEYWORDS) + 1
assert X_train_tab.shape[1] == len(CATEGORICAL_COLS) + n_title + len(EXPERIENCE_COLS)
assert X_train.shape[1] == X_train_tab.shape[1] + len(tfidf_feature_names)
print(f"Tabular features: {X_train_tab.shape[1]}")
print(f"TF-IDF features:  {len(tfidf_feature_names)}")
print(f"Total features:   {X_train.shape[1]}")

# NaN checks after bucketing: categoricals should now be 0 everywhere
for c in [f"{col}_enc" for col in CATEGORICAL_COLS]:
    n_null_val = X_val_tab[c].isna().sum()
    n_null_test = X_test_tab[c].isna().sum()
    print(f"{c}: val NaN={n_null_val} ({n_null_val/len(X_val_tab):.1%}), "
          f"test NaN={n_null_test} ({n_null_test/len(X_test_tab):.1%})")

for c in EXPERIENCE_COLS:
    n_null_train = X_train_tab[c].isna().sum()
    n_null_val = X_val_tab[c].isna().sum()
    n_null_test = X_test_tab[c].isna().sum()
    print(f"{c}: train NaN={n_null_train} ({n_null_train/len(X_train_tab):.1%}), "
          f"val NaN={n_null_val} ({n_null_val/len(X_val_tab):.1%}), "
          f"test NaN={n_null_test} ({n_null_test/len(X_test_tab):.1%})")

# Quick look at the corrected feature's distribution (train)
print("\nSeniority score distribution (train):")
print(X_train_tab["title_seniority_score"].value_counts().sort_index())

Tabular features: 36
TF-IDF features:  200
Total features:   236
category_celan_enc: val NaN=0 (0.0%), test NaN=0 (0.0%)
subcategory_celan_enc: val NaN=0 (0.0%), test NaN=0 (0.0%)
role_clean_enc: val NaN=0 (0.0%), test NaN=0 (0.0%)
state_enc: val NaN=0 (0.0%), test NaN=0 (0.0%)
type_enc: val NaN=0 (0.0%), test NaN=0 (0.0%)
exp_min: train NaN=7732 (37.8%), val NaN=1424 (38.9%), test NaN=1846 (39.3%)
exp_max: train NaN=12953 (63.3%), val NaN=2337 (63.9%), test NaN=2970 (63.2%)

Seniority score distribution (train):
title_seniority_score
0      640
1     2376
2    12299
3     2344
4     2592
5      143
6       68
Name: count, dtype: int64


In [56]:
print(X_train_tab.shape[1], "tabular columns:")
print(list(X_train_tab.columns))

36 tabular columns:
['category_celan_enc', 'subcategory_celan_enc', 'role_clean_enc', 'state_enc', 'type_enc', 'title_len', 'title_word_count', 'title_kw_senior', 'title_kw_lead', 'title_kw_head', 'title_kw_manager', 'title_kw_director', 'title_kw_executive', 'title_kw_chief', 'title_kw_vp', 'title_kw_vice_president', 'title_kw_principal', 'title_kw_junior', 'title_kw_intern', 'title_kw_entry', 'title_kw_gm', 'title_kw_controller', 'title_kw_ceo', 'title_kw_founder', 'title_kw_assistant', 'title_kw_clerk', 'title_kw_officer', 'title_kw_associate', 'title_kw_coordinator', 'title_kw_specialist', 'title_kw_supervisor', 'title_kw_analyst', 'title_kw_fresh', 'title_seniority_score', 'exp_min', 'exp_max']


In [57]:
# ----------------------------------------------------------------------
# 5. Target transform (log1p) — predicting (median, width) instead of
#    (min, width). Median as primary center target keeps this consistent
#    with the JobBERT range model's parameterization and with the
#    existing point-prediction MAE/R^2/acc_2k baseline.
# ----------------------------------------------------------------------
MIN_COL = "salary_min"
MAX_COL = "salary_max"
MID_COL = "salary_mid"
WIDTH_COL = "salary_width"  # precomputed — use directly instead of recomputing

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert (d[WIDTH_COL] >= 0).all(), f"Negative widths found in {name} — check inverted-range fix ran first."
    # Sanity: precomputed width should match max - min exactly
    assert np.allclose(d[WIDTH_COL], d[MAX_COL] - d[MIN_COL]), f"{name}: salary_width disagrees with max-min"

y_train_mid   = np.log1p(train_df[MID_COL].values)
y_train_width = np.log1p(train_df[WIDTH_COL].values)
y_val_mid     = np.log1p(val_df[MID_COL].values)
y_val_width   = np.log1p(val_df[WIDTH_COL].values)

# Test set kept in raw RM for evaluation (model predictions will need
# np.expm1 inverse-transform before comparing against these)
y_test_min_raw   = test_df[MIN_COL].values
y_test_max_raw   = test_df[MAX_COL].values
y_test_mid_raw   = test_df[MID_COL].values
y_test_width_raw = test_df[WIDTH_COL].values

In [58]:
# ----------------------------------------------------------------------
# 7. Fixed-config training run: edit the hyperparameter variables below.
#    Mid and width each have their own settings.
# ----------------------------------------------------------------------

# ===== HYPERPARAMETERS: edit here =====
# --- salary_mid model ---
MID_MAX_DEPTH         = 8
MID_LEARNING_RATE     = 0.025
MID_SUBSAMPLE         = 0.8
MID_COLSAMPLE_BYTREE  = 0.8
MID_MIN_CHILD_WEIGHT  = 1.0
MID_REG_LAMBDA        = 1.0

# --- salary_width model ---
WIDTH_MAX_DEPTH         = 4
WIDTH_LEARNING_RATE     = 0.025
WIDTH_SUBSAMPLE         = 0.8
WIDTH_COLSAMPLE_BYTREE  = 0.8
WIDTH_MIN_CHILD_WEIGHT  = 5
WIDTH_REG_LAMBDA        = 5

# --- shared ---
N_ESTIMATORS          = 1000
EARLY_STOPPING_ROUNDS = 30
# ======================================

BASE_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "enable_categorical": True,
    "random_state": SEED,
    "seed": SEED,
    "n_jobs": -1,
}

MID_PARAMS = {
    **BASE_PARAMS,
    "max_depth": MID_MAX_DEPTH,
    "learning_rate": MID_LEARNING_RATE,
    "subsample": MID_SUBSAMPLE,
    "colsample_bytree": MID_COLSAMPLE_BYTREE,
    "min_child_weight": MID_MIN_CHILD_WEIGHT,
    "reg_lambda": MID_REG_LAMBDA,
}

WIDTH_PARAMS = {
    **BASE_PARAMS,
    "max_depth": WIDTH_MAX_DEPTH,
    "learning_rate": WIDTH_LEARNING_RATE,
    "subsample": WIDTH_SUBSAMPLE,
    "colsample_bytree": WIDTH_COLSAMPLE_BYTREE,
    "min_child_weight": WIDTH_MIN_CHILD_WEIGHT,
    "reg_lambda": WIDTH_REG_LAMBDA,
}

def composite_score_on_val(pred_mid_log, pred_width_log, val_df):
    """Validation-set composite score: Hit Rate x calibration_factor,
    matching Section 6's best-model-selection criterion exactly, so
    hyperparameter decisions are judged on the same objective as the
    final model comparison."""
    pred_mid_raw = np.expm1(pred_mid_log)
    pred_width_raw = np.maximum(np.expm1(pred_width_log), 0)

    true_min = val_df[MIN_COL].values
    true_max = val_df[MAX_COL].values
    true_width = val_df[WIDTH_COL].values

    hit = (pred_mid_raw >= true_min) & (pred_mid_raw <= true_max)

    # Width ratio undefined when true_width == 0: exclude those rows from
    # the calibration factor rather than dividing by zero.
    nonzero_width = true_width > 0
    width_ratio = np.full_like(true_width, np.nan, dtype=float)
    width_ratio[nonzero_width] = pred_width_raw[nonzero_width] / true_width[nonzero_width]
    calibration_factor = np.minimum(width_ratio, 1.0 / width_ratio)

    composite = hit.astype(float) * calibration_factor
    return {
        "hit_rate": hit.mean(),
        "median_width_ratio": np.nanmedian(width_ratio),
        "composite_score_mean": np.nanmean(composite),
    }

model_mid = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    **MID_PARAMS,
)
model_mid.fit(
    X_train, y_train_mid,
    eval_set=[(X_val, y_val_mid)],
    verbose=False,
)

model_width = xgb.XGBRegressor(
    n_estimators=N_ESTIMATORS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    **WIDTH_PARAMS,
)
model_width.fit(
    X_train, y_train_width,
    eval_set=[(X_val, y_val_width)],
    verbose=False,
)

val_pred_mid_log = model_mid.predict(X_val)
val_pred_width_log = model_width.predict(X_val)

val_metrics = composite_score_on_val(val_pred_mid_log, val_pred_width_log, val_df)
val_mid_mae = mean_absolute_error(np.expm1(y_val_mid), np.expm1(val_pred_mid_log))
val_width_mae = mean_absolute_error(
    np.expm1(y_val_width), np.maximum(np.expm1(val_pred_width_log), 0)
)

print(f"MID:   depth={MID_MAX_DEPTH}, lr={MID_LEARNING_RATE}, subsample={MID_SUBSAMPLE}, "
      f"colsample={MID_COLSAMPLE_BYTREE}, mcw={MID_MIN_CHILD_WEIGHT}, lambda={MID_REG_LAMBDA}")
print(f"WIDTH: depth={WIDTH_MAX_DEPTH}, lr={WIDTH_LEARNING_RATE}, subsample={WIDTH_SUBSAMPLE}, "
      f"colsample={WIDTH_COLSAMPLE_BYTREE}, mcw={WIDTH_MIN_CHILD_WEIGHT}, lambda={WIDTH_REG_LAMBDA}")
print(f"best_iter_mid={model_mid.best_iteration}, best_iter_width={model_width.best_iteration}")
print(f"Val Hit Rate:          {val_metrics['hit_rate']:.4f}")
print(f"Val Median WidthRatio: {val_metrics['median_width_ratio']:.4f}")
print(f"Val Composite Score:   {val_metrics['composite_score_mean']:.4f}")
print(f"Val Mid MAE:           {val_mid_mae:,.0f}")
print(f"Val Width MAE:         {val_width_mae:,.0f}")

MID:   depth=8, lr=0.025, subsample=0.8, colsample=0.8, mcw=1.0, lambda=1.0
WIDTH: depth=4, lr=0.025, subsample=0.8, colsample=0.8, mcw=5, lambda=5
best_iter_mid=424, best_iter_width=148
Val Hit Rate:          0.4932
Val Median WidthRatio: 0.8737
Val Composite Score:   0.3704
Val Mid MAE:           904
Val Width MAE:         494


In [59]:
print(model_width.evals_result()['validation_0']['rmse'][:50])

[1.0588189761701834, 1.051014224454215, 1.0443794070350527, 1.0365897291594326, 1.029970586065874, 1.023505315764599, 1.0168975542287009, 1.0104698177788962, 1.0048717610755595, 0.9995982144424037, 0.9942550800840937, 0.9888103815875637, 0.9841920914087617, 0.9798560744524845, 0.9758971025921641, 0.9717359314411095, 0.9678501802700088, 0.9648140777202822, 0.960686069426279, 0.9572971045217266, 0.9539068379999026, 0.9506110946233355, 0.9471743302412553, 0.9445336062335626, 0.9413687351710746, 0.9382172181179878, 0.9357686206182002, 0.9335164686090673, 0.9306881459723928, 0.9284461593982757, 0.9258020500324816, 0.9241233895390815, 0.9224708537627185, 0.9205627845041441, 0.9190209838474271, 0.9173697923554525, 0.9158693591645948, 0.9140921625950157, 0.9125791802426164, 0.9114212973233521, 0.9101735366325434, 0.9091960794200312, 0.9081742720441504, 0.9072918010697346, 0.9058310201814931, 0.9044245602632638, 0.9033015192736176, 0.902414677281343, 0.9011942294763662, 0.9005134096276878]


In [60]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def fit_bracket_edges(train_mid, n_brackets=4):
    """Quartile edges for salary_mid, fit on TRAIN ONLY."""
    quantiles = np.linspace(0, 1, n_brackets + 1)
    edges = np.quantile(train_mid, quantiles)
    edges[0] = -np.inf
    edges[-1] = np.inf

    labels = []
    for i in range(n_brackets):
        lo, hi = edges[i], edges[i + 1]
        lo_str = "0" if lo == -np.inf else f"{lo:,.0f}"
        hi_str = "∞" if hi == np.inf else f"{hi:,.0f}"
        labels.append(f"RM{lo_str}–{hi_str}")
    return edges, labels


def fit_width_bucket_edges(train_true_width, n_buckets=3):
    """Tercile edges for true width, fit on TRAIN ONLY, excluding zero-width rows."""
    nonzero = train_true_width[train_true_width > 0]
    quantiles = np.linspace(0, 1, n_buckets + 1)
    edges = np.quantile(nonzero, quantiles)
    edges[0] = 0
    edges[-1] = np.inf

    names = ["narrow", "medium", "wide"] if n_buckets == 3 else [f"bucket_{i+1}" for i in range(n_buckets)]
    labels = [f"{names[i]} (RM{edges[i]:,.0f}–{'∞' if edges[i+1]==np.inf else f'{edges[i+1]:,.0f}'})" for i in range(n_buckets)]
    return edges, labels


def assign_bracket(mid_values, edges, labels):
    idx = np.searchsorted(edges, mid_values, side="right") - 1
    idx = np.clip(idx, 0, len(labels) - 1)
    return np.array(labels)[idx]


def assign_width_bucket(true_width_values, edges, labels):
    """Zero-width rows get a dedicated 'zero_width' label."""
    out = np.empty(len(true_width_values), dtype=object)
    zero_mask = true_width_values == 0
    out[zero_mask] = "zero_width"

    nonzero_vals = true_width_values[~zero_mask]
    idx = np.searchsorted(edges, nonzero_vals, side="right") - 1
    idx = np.clip(idx, 0, len(labels) - 1)
    out[~zero_mask] = np.array(labels)[idx]
    return out


def layer1_component_metrics(true_mid, pred_mid, true_width, pred_width):
    def _block(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan

        y_true_log = np.log1p(np.maximum(y_true, 0))
        y_pred_log = np.log1p(np.maximum(y_pred, 0))
        r2_log = r2_score(y_true_log, y_pred_log) if len(y_true) > 1 else np.nan

        return {"MAE": mae, "RMSE": rmse, "R2": r2, "R2_log": r2_log, "n": len(y_true)}
    return {"mid": _block(true_mid, pred_mid), "width": _block(true_width, pred_width)}


def layer2_hit_rate(true_min, true_max, pred_mid):
    hit = (pred_mid >= true_min) & (pred_mid <= true_max)
    result = {"hit_rate": hit.mean(), "n": len(pred_mid)}

    miss_mask = ~hit
    if miss_mask.sum() > 0:
        dist_below = np.maximum(true_min[miss_mask] - pred_mid[miss_mask], 0)
        dist_above = np.maximum(pred_mid[miss_mask] - true_max[miss_mask], 0)
        dtb = np.maximum(dist_below, dist_above)
        result["dtb_mae"] = float(dtb.mean())
        result["dtb_n"] = int(miss_mask.sum())
    else:
        result["dtb_mae"] = np.nan
        result["dtb_n"] = 0

    return result


def composite_score(true_min, true_max, true_width, pred_mid, pred_width):
    """Section 6 best-model-selection score: Hit Rate x calibration_factor,
    calibration_factor = min(width_ratio, 1/width_ratio). Zero-width rows
    are excluded from the calibration factor (undefined ratio) but still
    count toward Hit Rate."""
    hit = (pred_mid >= true_min) & (pred_mid <= true_max)

    true_width = np.asarray(true_width)
    pred_width = np.asarray(pred_width)
    zero_mask = true_width == 0

    calibration_factor = np.full(len(true_width), np.nan, dtype=float)
    nonzero_ratio = pred_width[~zero_mask] / true_width[~zero_mask]
    calibration_factor[~zero_mask] = np.minimum(nonzero_ratio, 1.0 / nonzero_ratio)

    per_row_composite = hit.astype(float) * calibration_factor

    return {
        "composite_mean": np.nanmean(per_row_composite),
        "composite_n": int((~zero_mask).sum()),
        "hit_rate": hit.mean(),
        "median_calibration_factor": np.nanmedian(calibration_factor),
        "n_zero_width_excluded": int(zero_mask.sum()),
    }


def layer3_width_ratio(true_width, pred_width):
    zero_mask = true_width == 0
    nonzero_true = true_width[~zero_mask]
    nonzero_pred = pred_width[~zero_mask]
    ratio = nonzero_pred / nonzero_true

    result = {
        "median_ratio": np.median(ratio),
        "q1_ratio": np.percentile(ratio, 25),
        "q3_ratio": np.percentile(ratio, 75),
        "n": len(ratio),
    }
    zero_width_pred = pred_width[zero_mask]
    if len(zero_width_pred) > 0:
        result["zero_width_median_pred_width"] = np.median(zero_width_pred)
        result["zero_width_n"] = len(zero_width_pred)
    return result


def full_metrics_report(df, bracket_edges, bracket_labels, width_edges, width_labels):
    df = df.copy()
    df["bracket"] = assign_bracket(df["true_mid"].values, bracket_edges, bracket_labels)
    df["width_bucket"] = assign_width_bucket(df["true_width"].values, width_edges, width_labels)

    report = {"overall": {}, "by_bracket": {}, "by_width_bucket": {}}

    report["overall"]["layer1"] = layer1_component_metrics(
        df["true_mid"], df["pred_mid"], df["true_width"], df["pred_width"])
    report["overall"]["layer2"] = layer2_hit_rate(df["true_min"], df["true_max"], df["pred_mid"])
    report["overall"]["layer3"] = layer3_width_ratio(df["true_width"].values, df["pred_width"].values)
    report["overall"]["composite"] = composite_score(
        df["true_min"].values, df["true_max"].values, df["true_width"].values,
        df["pred_mid"].values, df["pred_width"].values)

    for b in bracket_labels:
        sub = df[df["bracket"] == b]
        if len(sub) == 0:
            continue
        report["by_bracket"][b] = {
            "layer1": layer1_component_metrics(
                sub["true_mid"], sub["pred_mid"], sub["true_width"], sub["pred_width"]),
            "layer2": layer2_hit_rate(sub["true_min"], sub["true_max"], sub["pred_mid"]),
            "layer3": layer3_width_ratio(sub["true_width"].values, sub["pred_width"].values),
            "composite": composite_score(
                sub["true_min"].values, sub["true_max"].values, sub["true_width"].values,
                sub["pred_mid"].values, sub["pred_width"].values),
        }

    for wb in width_labels + ["zero_width"]:
        sub = df[df["width_bucket"] == wb]
        if len(sub) == 0:
            continue
        report["by_width_bucket"][wb] = layer2_hit_rate(sub["true_min"], sub["true_max"], sub["pred_mid"])

    return report


def print_report(report, title="Metrics Report"):
    print(f"\n{'='*60}\n{title}\n{'='*60}")

    print("\n--- Overall ---")
    l1 = report["overall"]["layer1"]
    print(f"  Mid:   MAE={l1['mid']['MAE']:,.0f}  RMSE={l1['mid']['RMSE']:,.0f}  "
      f"R2={l1['mid']['R2']:.3f}  R2_log={l1['mid']['R2_log']:.3f}")
    print(f"  Width: MAE={l1['width']['MAE']:,.0f}  RMSE={l1['width']['RMSE']:,.0f}  "
      f"R2={l1['width']['R2']:.3f}  R2_log={l1['width']['R2_log']:.3f}")
    l2 = report["overall"]["layer2"]
    print(f"  Hit Rate: {l2['hit_rate']:.3f} (n={l2['n']})")
    if l2["dtb_n"] > 0:
        print(f"  DtB-MAE (misses only): RM {l2['dtb_mae']:,.0f} (n={l2['dtb_n']})")
    l3 = report["overall"]["layer3"]
    print(f"  Width Ratio: median={l3['median_ratio']:.2f}  IQR=[{l3['q1_ratio']:.2f}, {l3['q3_ratio']:.2f}]")
    if "zero_width_median_pred_width" in l3:
        print(f"  Zero-width true rows (n={l3['zero_width_n']}): median pred_width={l3['zero_width_median_pred_width']:,.0f}")
    comp = report["overall"]["composite"]
    print(f"  Composite Score: {comp['composite_mean']:.3f} (n={comp['composite_n']}, "
          f"{comp['n_zero_width_excluded']} zero-width rows excluded)")

    print("\n--- By Salary Bracket ---")
    for b, m in report["by_bracket"].items():
        dtb_str = f"DtB={m['layer2']['dtb_mae']:,.0f}" if m['layer2']['dtb_n'] > 0 else "DtB=n/a"
        print(f"  [{b}] n={m['layer1']['mid']['n']}  MidMAE={m['layer1']['mid']['MAE']:,.0f}  "
              f"HitRate={m['layer2']['hit_rate']:.3f}  {dtb_str}  "
              f"WidthRatio(med)={m['layer3']['median_ratio']:.2f}  "
              f"Composite={m['composite']['composite_mean']:.3f}")

    print("\n--- By True-Width Bucket (Hit Rate + DtB-MAE) ---")
    for wb, m in report["by_width_bucket"].items():
        dtb_str = f"DtB={m['dtb_mae']:,.0f}" if m['dtb_n'] > 0 else "DtB=n/a"
        print(f"  [{wb}] n={m['n']}  HitRate={m['hit_rate']:.3f}  {dtb_str}")

In [61]:
# ----------------------------------------------------------------------
# Supplementary diagnostic: fixed-width RM2,000 salary brackets.
# Purpose: finer-grained view of WHERE performance degrades, since the
# quartile-based primary brackets collapse everything above ~RM5,150 into
# one catch-all bucket. This is diagnostic only — quartile brackets remain
# the primary reported bracket scheme per metrics_framework.md.
# ----------------------------------------------------------------------
def fit_width_bracket_edges(max_salary, bin_width=2000):
    n_bins = int(np.ceil(max_salary / bin_width))
    edges = np.arange(0, (n_bins + 1) * bin_width, bin_width, dtype=float)
    edges[0] = -np.inf
    edges[-1] = np.inf  # catch anything above the last full bin

    labels = []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        lo_str = "0" if lo == -np.inf else f"{lo:,.0f}"
        hi_str = "∞" if hi == np.inf else f"{hi:,.0f}"
        labels.append(f"RM{lo_str}–{hi_str}")
    return edges, labels

# Fit on train, same discipline as the primary brackets
fine_bracket_edges, fine_bracket_labels = fit_width_bracket_edges(
    train_df[MID_COL].max(), bin_width=2000
)



In [62]:
# ----------------------------------------------------------------------
# 7c. Final test-set evaluation — single fixed config (max_depth=8,
#     learning_rate=0.05), using the existing model_mid/model_width from
#     the earlier fixed-config training cell. No selection step needed
#     since only one configuration is being evaluated.
# ----------------------------------------------------------------------

bracket_edges, bracket_labels = fit_width_bracket_edges(
    train_df[MID_COL].max(), bin_width=2000
)
width_edges, width_labels = fit_width_bucket_edges(train_df[WIDTH_COL].values, n_buckets=3)

pred_mid_raw = np.expm1(model_mid.predict(X_test))
pred_width_raw = np.maximum(np.expm1(model_width.predict(X_test)), 0)

eval_df_final = pd.DataFrame({
    "true_mid": y_test_mid_raw,
    "pred_mid": pred_mid_raw,
    "true_width": y_test_width_raw,
    "pred_width": pred_width_raw,
    "true_min": y_test_min_raw,
    "true_max": y_test_max_raw,
})

final_report = full_metrics_report(eval_df_final, bracket_edges, bracket_labels, width_edges, width_labels)
print_report(final_report, title=f"XGBoost Final Test Evaluation seed={SEED}")


XGBoost Final Test Evaluation seed=42

--- Overall ---
  Mid:   MAE=987  RMSE=1,619  R2=0.613  R2_log=0.675
  Width: MAE=531  RMSE=807  R2=0.304  R2_log=0.222
  Hit Rate: 0.450 (n=4700)
  DtB-MAE (misses only): RM 904 (n=2586)
  Width Ratio: median=0.90  IQR=[0.66, 1.31]
  Zero-width true rows (n=121): median pred_width=654
  Composite Score: 0.341 (n=4579, 121 zero-width rows excluded)

--- By Salary Bracket ---
  [RM0–2,000] n=326  MidMAE=746  HitRate=0.141  DtB=740  WidthRatio(med)=1.99  Composite=0.064
  [RM2,000–4,000] n=2228  MidMAE=587  HitRate=0.489  DtB=576  WidthRatio(med)=1.04  Composite=0.379
  [RM4,000–6,000] n=1240  MidMAE=782  HitRate=0.537  DtB=607  WidthRatio(med)=0.83  Composite=0.400
  [RM6,000–8,000] n=450  MidMAE=1,397  HitRate=0.393  DtB=972  WidthRatio(med)=0.72  Composite=0.292
  [RM8,000–10,000] n=214  MidMAE=2,174  HitRate=0.374  DtB=1,903  WidthRatio(med)=0.72  Composite=0.288
  [RM10,000–12,000] n=122  MidMAE=2,889  HitRate=0.328  DtB=2,225  WidthRatio(med)

In [63]:
bracket_edges_2k, bracket_labels_2k = fit_width_bracket_edges(
    train_df[MID_COL].max(), bin_width=2000
)

for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    brackets = assign_bracket(d[MID_COL].values, bracket_edges_2k, bracket_labels_2k)
    counts = pd.Series(brackets).value_counts().reindex(bracket_labels_2k, fill_value=0)
    print(f"\n--- {name} (n={len(d)}) ---")
    print(counts.to_string())


--- Train (n=20462) ---
RM0–2,000          1025
RM2,000–4,000      9623
RM4,000–6,000      5697
RM6,000–8,000      2135
RM8,000–10,000      895
RM10,000–12,000     512
RM12,000–14,000     284
RM14,000–16,000     140
RM16,000–18,000     108
RM18,000–∞           43

--- Val (n=3660) ---
RM0–2,000           207
RM2,000–4,000      1763
RM4,000–6,000      1058
RM6,000–8,000       335
RM8,000–10,000      146
RM10,000–12,000      78
RM12,000–14,000      36
RM14,000–16,000      15
RM16,000–18,000      16
RM18,000–∞            6

--- Test (n=4700) ---
RM0–2,000           326
RM2,000–4,000      2228
RM4,000–6,000      1240
RM6,000–8,000       450
RM8,000–10,000      214
RM10,000–12,000     122
RM12,000–14,000      61
RM14,000–16,000      22
RM16,000–18,000      29
RM18,000–∞            8
